# EnergyFlow.jl — EMD Example
Compute Earth Mover's Distance between particle physics events.

In [ ]:
# Load package and events
using Pkg
Pkg.activate(@__DIR__)
Pkg.develop(path=joinpath(@__DIR__, "..", ".."))
Pkg.instantiate()

using EnergyFlow

events = load_hepmc3_events(joinpath(@__DIR__, "..", "..", "data", "sk_example_PU.hepmc"); maxevents=20)
println("Loaded $(length(events)) events")

Loaded 20 events


In [2]:
# ── Single EMD ──

# Default: ns64 backend, Euclidean distance
val = emd(events[1], events[2]; R=1.0, beta=1.0, norm=true)
println("EMD (ns64, Euclidean): $val")

# ot64 backend, EtaPhi distance
val2 = emd(events[1], events[2]; R=1.0, beta=1.0, norm=true, backend=:ot64, metric=EtaPhiMetric())
println("EMD (ot64, EtaPhi):    $val2")

# Other backends: :ns32, :ot32 (Float32), :sinkhorn (approximate)
# Other metrics:  SquaredEuclideanMetric(), PrecomputedMetric(matrix), CustomMetric(f)

EMD (ns64, Euclidean): 1.8170644797464628
EMD (ot64, EtaPhi):    1.771341959192871


In [3]:
# ── Pairwise EMD ──

# Cross-pairwise: 5 vs 5 → 5×5 matrix
D = emds(events[1:5], events[5:10]; R=1.0, beta=1.0, norm=true)
println("Cross-pairwise (5×5): D[1,1]=$(D[1,1])")

# Self-pairwise: flat upper-triangular vector
dists = emds(events[1:5]; R=1.0, beta=1.0, norm=true)
println("Self-pairwise ($(length(dists)) pairs): dists[1]=$(dists[1])")

dists_verify = emds(events[1:5], events[1:5]; R=1.0, beta=1.0, norm=true)
println("Self-pairwise verification (5×5): dists_verify[2,1]=$(dists_verify[2,1])")

# With EtaPhi metric
D2 = emds(events[1:5], events[6:10]; R=1.0, beta=1.0, norm=true, metric=EtaPhiMetric())
println("Cross EtaPhi: D[1,1]=$(D2[1,1])")

Cross-pairwise (5×5): D[1,1]=1.844030340864774
Self-pairwise (10 pairs): dists[1]=1.8170644797464628
Self-pairwise verification (5×5): dists_verify[2,1]=1.8170644797464632
Cross EtaPhi: D[1,1]=2.037071965495491


In [9]:
# Print dists in a triangular shape
println("Self-pairwise distances (dists, triangular):")
if ndims(dists) == 1
    m = length(dists)
    n = Int(floor((sqrt(8m + 1) - 1) / 2))
    if n * (n + 1) ÷ 2 == m
        idx = 1
        for i in 1:n
            println(join(dists[idx:idx + i - 1], " "))
            idx += i
        end
    else
        println(dists)
    end
else
    nrows, ncols = size(dists)
    for i in 1:nrows
        println(join((dists[j, i] for j in 1:min(i, ncols)), " "))
    end
end

# Print dists_verify with a different line setup (one row per line)
println("\nCross-pairwise distances with Euclidean metric (dists_verify, row-by-row):")
for i in 1:size(dists_verify, 1)
    println(dists_verify[i, :])
end

Self-pairwise distances (dists, triangular):
1.8170644797464628
2.040338354518915 1.6435577823620169
1.844030340864774 1.0717332097453254 1.0864205767377015
1.2513706519208507 1.24567333306541 1.221765764826129 0.9256947314958315

Cross-pairwise distances with Euclidean metric (dists_verify, row-by-row):
[0.0, 1.8170644797464628, 2.040338354518915, 1.6435577823620169, 1.844030340864774]
[1.8170644797464632, 0.0, 1.0717332097453254, 1.0864205767377015, 1.2513706519208507]
[2.040338354518915, 1.0717332097453245, 0.0, 1.24567333306541, 1.221765764826129]
[1.6435577823620167, 1.086420576737702, 1.24567333306541, 0.0, 0.9256947314958315]
[1.844030340864774, 1.2513706519208494, 1.221765764826129, 0.9256947314958315, 0.0]
